# Libraries

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score
from itertools import combinations


a) correlation coefficients

In [4]:

#Loadining dataset
college=pd.read_csv('College.csv')

#Extracting necessary columns
college_new=college[['Apps','Enroll','Outstate','Top10perc','Top25perc','Grad.Rate']]

# Calculating correlation coefficients
correlation_matrix = college_new.corr()
correlation_with_grad_rate = correlation_matrix['Grad.Rate'].drop(labels=['Grad.Rate'])
print('Correlation coefficients with Grad.Rate:')
print(correlation_with_grad_rate)

Correlation coefficients with Grad.Rate:
Apps         0.146755
Enroll      -0.022341
Outstate     0.571290
Top10perc    0.494989
Top25perc    0.477281
Name: Grad.Rate, dtype: float64


b) Using foward to build regression model

Stepwise Selection:stepwise regression involves iteratively adding predictors based on a selection criterion ( p-value). The stepwise process identifies the predictors that have the most significant contribution to explaining the graduation rate.

In [5]:

def forward_regression(X, y, threshold_in, verbose=True):
    initial_list = []
    included = list(initial_list)
    
    while True:
        changed = False
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded)

        # Calculate and print p-values for each excluded variable
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_column]]))).fit()
            new_pval[new_column] = model.pvalues[new_column]
        
        # Print the p-values for the current step
        print("\nP-values for current step:")
        print(new_pval)
        
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval))
        
        if not changed:
            break
    
    final_model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
    print(final_model.summary())
    
    # Print p-values of the final model
    print("\nP-values of the final model's features:")
    print(final_model.pvalues)
    
    return included

X = college_new[['Apps', 'Enroll', 'Outstate', 'Top10perc', 'Top25perc']]
y = college['Grad.Rate']
included = forward_regression(X, y, 0.05, verbose=False)



P-values for current step:
Apps         4.018556e-05
Top25perc    1.872333e-45
Outstate     1.628927e-68
Top10perc    2.897974e-49
Enroll       5.340568e-01
dtype: float64

P-values for current step:
Top10perc    4.471230e-13
Enroll       2.234948e-02
Top25perc    4.695033e-15
Apps         5.625656e-05
dtype: float64

P-values for current step:
Top10perc    0.270388
Enroll       0.640880
Apps         0.207867
dtype: float64
                            OLS Regression Results                            
Dep. Variable:              Grad.Rate   R-squared:                       0.378
Model:                            OLS   Adj. R-squared:                  0.376
Method:                 Least Squares   F-statistic:                     235.0
Date:                Tue, 15 Oct 2024   Prob (F-statistic):           1.82e-80
Time:                        20:01:02   Log-Likelihood:                -3127.2
No. Observations:                 777   AIC:                             6260.
Df Residuals:     

c) use full predictors

In [6]:
print("selected features: ", included)

selected features:  ['Outstate', 'Top25perc']


d) using  BIC to to select best features to build the model

In [7]:

# Features and target
X = ['Apps', 'Enroll', 'Outstate', 'Top10perc', 'Top25perc']
y = college_new['Grad.Rate']

# Initialize dictionary to store BIC scores for each model
bic_scores = {}

# Generate all possible non-empty combinations of the features
for r in range(1, len(X) + 1):
    for comb in combinations(X, r):
        # Convert tuple to list for indexing purposes
        feature_subset = list(comb)

        # Prepare the feature subset for the model
        X_subset = college_new[feature_subset]
        X_subset = sm.add_constant(X_subset)  # Add a constant for the intercept

        # Fit the model
        model = sm.OLS(y, X_subset).fit()

        # Calculate log-likelihood, number of parameters, and number of samples
        log_likelihood = model.llf
        n_params = model.df_model + 1  # Number of parameters (including intercept)
        n_samples = len(y)

        # Calculate BIC
        bic = -2 * log_likelihood + np.log(n_samples) * n_params
        bic_scores[comb] = bic

# Find the model with the lowest BIC
best_model = min(bic_scores, key=bic_scores.get)

# Display BIC scores for all models
print("BIC Scores for each model:")
for model, bic in bic_scores.items():
    print(f"{model}: {bic}")

# Display the best model
print(f"\nBest Model (lowest BIC): {best_model}")


BIC Scores for each model:
('Apps',): 6619.397696622621
('Enroll',): 6635.926794151646
('Outstate',): 6329.339476057561
('Top10perc',): 6417.933787526683
('Top25perc',): 6435.453828484054
('Apps', 'Enroll'): 6563.23939898376
('Apps', 'Outstate'): 6319.696847637415
('Apps', 'Top10perc'): 6424.078071616602
('Apps', 'Top25perc'): 6441.59905528857
('Enroll', 'Outstate'): 6330.752936274306
('Enroll', 'Top10perc'): 6411.105481565276
('Enroll', 'Top25perc'): 6423.813497284463
('Outstate', 'Top10perc'): 6283.333457513429
('Outstate', 'Top25perc'): 6274.3329824422635
('Top10perc', 'Top25perc'): 6418.124429160247
('Apps', 'Enroll', 'Outstate'): 6320.507268021144
('Apps', 'Enroll', 'Top10perc'): 6396.704915207951
('Apps', 'Enroll', 'Top25perc'): 6401.190886336495
('Apps', 'Outstate', 'Top10perc'): 6287.7736961743685
('Apps', 'Outstate', 'Top25perc'): 6279.392964749951
('Apps', 'Top10perc', 'Top25perc'): 6423.737387068755
('Enroll', 'Outstate', 'Top10perc'): 6289.984325613547
('Enroll', 'Outstate'

Bayesian Information Criterion (BIC): BIC was used to assess the performance of different models with varying combinations of features. It penalizes models with more predictors, thus balancing model complexity with goodness-of-fit. The model with the lowest BIC score was selected, indicating that it is the most parsimonious (simplest) model that best fits the data.

Since both stepwise selection and BIC gave the same set of predictor variables, it confirms that these features are the most useful in predicting the graduation rate.

e) comparing the accuracy of the models using useful predictors and all predictors

Model 1: Useful Predictors (based on stepwise or BIC selection)

In [8]:

useful_predictors = ['Outstate', 'Top25perc'] 
X_useful = college_new[useful_predictors]
X_useful = sm.add_constant(X_useful)
model_useful = sm.OLS(y, X_useful).fit()

y_pred_useful = model_useful.predict(X_useful)

# Calculate metrics
mape_useful = mean_absolute_percentage_error(y, y_pred_useful)
rmse_useful = root_mean_squared_error(y, y_pred_useful)
r2_useful = model_useful.rsquared

print( f"Model 1: Useful Predictors\
      \nMAPE: {mape_useful}\
      \nRMSE: {rmse_useful}\
      \nR^2: {r2_useful}")

Model 1: Useful Predictors      
MAPE: 0.19212075643794257      
RMSE: 13.541383460670172      
R^2: 0.37776441749868717


Model 2: All Predictors

In [9]:
X = ['Apps', 'Enroll', 'Outstate', 'Top10perc', 'Top25perc']
X_all = college_new[X] 
X_all = sm.add_constant(X_all)
model_all = sm.OLS(y, X_all).fit()

y_pred_all = model_all.predict(X_all)

# Calculate metrics
mape_all = mean_absolute_percentage_error(y, y_pred_all)
rmse_all = root_mean_squared_error(y, y_pred_all)
r2_all = model_all.rsquared

print (f"Model 2: All Predictors\
      \nMAPE: {mape_all}\
      \nRMSE: {rmse_all}\
      \nR^2: {r2_all}")

Model 2: All Predictors      
MAPE: 0.1905128818178193      
RMSE: 13.449738618193683      
R^2: 0.3861582005130557


Comparison

In [10]:
# Compare RMSE, MAPE, and R-squared values one by one
if rmse_useful < rmse_all:
    print("Model 1 (Useful Predictors) has a lower RMSE than Model 2 (All Predictors).")
elif rmse_all < rmse_useful:
    print("Model 2 (All Predictors) has a lower RMSE than Model 1 (Useful Predictors).")
else:
    print("Both models have the same RMSE.")

if mape_useful < mape_all:
    print("Model 1 (Useful Predictors) has a lower MAPE than Model 2 (All Predictors).")
elif mape_all < mape_useful:
    print("Model 2 (All Predictors) has a lower MAPE than Model 1 (Useful Predictors).")
else:
    print("Both models have the same MAPE.")

if r2_useful > r2_all:
    print("Model 1 (Useful Predictors) has a higher R-squared than Model 2 (All Predictors).")
elif r2_all > r2_useful:
    print("Model 2 (All Predictors) has a higher R-squared than Model 1 (Useful Predictors).")
else:
    print("Both models have the same R-squared.")

# Final decision based on overall comparison
if rmse_useful < rmse_all and mape_useful < mape_all and r2_useful > r2_all:
    print("\nModel 1 (Useful Predictors) is the better model overall.")
elif rmse_all < rmse_useful and mape_all < mape_useful and r2_all > r2_useful:
    print("\nModel 2 (All Predictors) is the better model overall.")
else:
    print("\nThe models have mixed performance. Consider trade-offs based on specific metrics.")

Model 2 (All Predictors) has a lower RMSE than Model 1 (Useful Predictors).
Model 2 (All Predictors) has a lower MAPE than Model 1 (Useful Predictors).
Model 2 (All Predictors) has a higher R-squared than Model 1 (Useful Predictors).

Model 2 (All Predictors) is the better model overall.


f) Predicting CMU graduation rate using the best model

In [11]:

# Filter data for Carnegie Mellon University
cmu_data = college[college['Unnamed: 0'] == 'Carnegie Mellon University'].reset_index()
cmu_predictors = cmu_data[['Apps', 'Enroll', 'Outstate', 'Top10perc', 'Top25perc']]
cmu_predictors = sm.add_constant(cmu_predictors, has_constant='add')

# Predict graduation rate using the most accurate model(all features)
graduation_rate_pred = model_all.predict(cmu_predictors)

#predict using only useful predictors
#graduation_rate_pred2=model_useful.predict(X_useful)

print(f"Predicted graduation rate for Carnegie Mellon University: {graduation_rate_pred[0]:.2f}")
#print( f"Predicted graduation rate for Carnegie Mellon University: {graduation_rate_pred2[0]:.2f}")

Predicted graduation rate for Carnegie Mellon University: 89.20
